# 02 · Previsão mensal com LightGBM e Optuna

Execute o Notebook 01 antes deste. Os últimos cinco meses são reservados pelo
mês-alvo. Os dois primeiros guiam a otimização e os três seguintes medem o erro.
Todos os meses anteriores entram no treino sempre que houver observações e
defasagens suficientes. Instale as dependências de desenvolvimento para usar o Optuna.

A busca compara uma configuração por variável, compartilhada pelos horizontes.
O corte de treino da validação também respeita o mês de origem de cada previsão.

## Configurar caminhos

In [5]:
import sys
from pathlib import Path

import pandas as pd

ROOT = next(
    (
        path
        for path in (Path.cwd(), *Path.cwd().parents)
        if (path / "src/feynmans_legacy/forecast_tuning.py").is_file()
    ),
    None,
)
if ROOT is None:
    raise FileNotFoundError("Abra o notebook a partir do repositório feynmans-legacy.")

sys.path.insert(0, str(ROOT / "src"))

TERRA = ROOT / "data" / "terra"
RUN = TERRA / "analise_analogos"
terra_path = TERRA / "terra.parquet"

## Definir alvos, horizontes e amostra de treino

O ajuste e a validação cronológica usam uma amostra reproduzível de locais.
A inferência usa todos os locais com defasagens suficientes no mês de corte.

import json
import pickle

from feynmans_legacy.analog_study import LAGS, publish_forecasts
from feynmans_legacy.forecast_tuning import train_forecasts

HORIZONTES = [1, 2, 3]
LOCAIS_TREINO = 400
SEMENTE = 42
TENTATIVAS_OPTUNA = 24
selection = json.loads((RUN / "selecao.json").read_text(encoding="utf-8"))
TARGETS = selection["alvos_previsao"]
terra_path = TERRA / "terra.parquet"
locations = pd.read_parquet(RUN / "coordenadas.parquet", columns=["location_id"])
sample = locations["location_id"].sort_values().sample(
    n=min(LOCAIS_TREINO, len(locations)), random_state=SEMENTE
)
history = pd.read_parquet(
    terra_path, columns=["location_id", "anomes", *TARGETS],
    filters=[("location_id", "in", sample.tolist())],
)
sample_size = history["location_id"].nunique()
print(f"Treino com {sample_size:,} locais e {len(TARGETS)} variáveis dinâmicas")

In [6]:
import json
import pickle

from feynmans_legacy.analog_study import LAGS, publish_forecasts

HORIZONTES = [1, 2, 3]
TENTATIVAS_OPTUNA = 24
LOCAIS_TREINO = 400
SEMENTE = 42

selecao_path = RUN / "selecao.json"
coordenadas_path = RUN / "coordenadas.parquet"

ausentes = [
    str(path.relative_to(ROOT))
    for path in (terra_path, selecao_path, coordenadas_path)
    if not path.is_file()
]
if ausentes:
    raise FileNotFoundError(
        f"Arquivos ausentes: {', '.join(ausentes)}. Confira a execução do Notebook 01."
    )

TARGETS = json.loads(selecao_path.read_text(encoding="utf-8"))["alvos_previsao"]
locations = pd.read_parquet(coordenadas_path, columns=["location_id"])
ids = locations["location_id"].drop_duplicates().sort_values()
sample = ids.sample(n=min(LOCAIS_TREINO, len(ids)), random_state=SEMENTE)

history = pd.read_parquet(
    terra_path,
    columns=["location_id", "anomes", *TARGETS],
    filters=[("location_id", "in", sample.tolist())],
)
sample_size = history["location_id"].nunique()

print(
    f"Treino com {sample_size:,} locais, {len(history):,} linhas "
    f"e {len(TARGETS)} variáveis dinâmicas."
)

Treino com 400 locais, 36,800 linhas e 9 variáveis dinâmicas.


## Otimizar na validação e medir os três meses de teste

O Optuna minimiza o MAE médio dos dois meses de validação e dos horizontes
disponíveis, na unidade da variável. A busca limita a complexidade das árvores,
avalia regularização e compara o resultado com o LightGBM padrão. O método
publicado por horizonte é escolhido pelo MAE de validação, incluindo persistência.

O teste usa o modelo ajustado apenas nos meses de treino. As previsões com origem
anterior ao fim da validação também são contabilizadas como retrospectivas. As
colunas operacionais medem somente previsões que poderiam usar os parâmetros
selecionados naquela data. Após medir o teste, o modelo de produção é reajustado
com as observações disponíveis até o último mês para publicar os meses futuros.

In [7]:
from time import perf_counter
from unittest.mock import patch

import numpy as np
import optuna

import feynmans_legacy.forecast_tuning as tuning


ESPACO_BUSCA = {
    "objective": ("categorical", {
        "choices": ["regression_l2", "regression_l1", "huber", "fair"],
    }),
    "boosting_type": ("categorical", {"choices": ["gbdt", "dart"]}),
    "n_estimators": ("int", {"low": 80, "high": 320, "step": 20}),
    "learning_rate": ("float", {"low": 0.03, "high": 0.12, "log": True}),
    "max_depth": ("int", {"low": 3, "high": 6}),
    "num_leaves": ("int", {"low": 4, "high": 64}),
    "min_child_samples": ("int", {"low": 20, "high": 150, "step": 5}),
    "min_child_weight": ("float", {"low": 1e-3, "high": 10.0, "log": True}),
    "min_split_gain": ("float", {"low": 0.0, "high": 1.0}),
    "reg_alpha": ("float", {"low": 0.0, "high": 5.0}),
    "reg_lambda": ("float", {"low": 0.0, "high": 10.0}),
    "max_bin": ("categorical", {"choices": [63, 127, 255]}),
    "min_data_in_bin": ("int", {"low": 3, "high": 30}),
    "colsample_bytree": ("float", {"low": 0.65, "high": 1.0}),
    "subsample_freq": ("categorical", {"choices": [0, 1, 2, 4]}),
    "subsample": ("float", {"low": 0.65, "high": 1.0}),
}


def sugerir_parametros(trial):
    params = {}

    for nome, (tipo, limites) in ESPACO_BUSCA.items():
        if nome == "data_sample_strategy" and params["boosting_type"] == "dart":
            params[nome] = "bagging"
            continue

        if nome == "subsample_freq" and params["data_sample_strategy"] != "bagging":
            continue
        if nome == "subsample" and params.get("subsample_freq", 0) == 0:
            continue

        kwargs = dict(limites)
        if nome == "num_leaves":
            kwargs["high"] = min(kwargs["high"], 2 ** params["max_depth"])
            if kwargs["low"] > kwargs["high"]:
                raise ValueError("O mínimo de folhas excede o limite da profundidade.")

        sugerir = getattr(trial, f"suggest_{tipo}")
        params[nome] = sugerir(nome, **kwargs)

    return params


def otimizar_com_log(splits, n_trials, seed):
    horizontes = [split.horizon for split in splits]
    mae_padrao = [
        tuning._validation_mae(split.validation, tuning.DEFAULT_PARAMS, seed)
        for split in splits
    ]
    mae_medio_padrao = float(np.mean(mae_padrao))
    print(f"LightGBM padrão | MAE validação={mae_medio_padrao:.6g}", flush=True)

    def objetivo(trial):
        inicio = perf_counter()
        try:
            params = sugerir_parametros(trial)
            trial.set_user_attr("parametros_completos", params)

            erros = [
                tuning._validation_mae(split.validation, params, seed)
                for split in splits
            ]
            trial.set_user_attr("mae_por_horizonte", erros)
            return float(np.mean(erros))
        finally:
            trial.set_user_attr("duracao_s", perf_counter() - inicio)

    def registrar_trial(study, trial):
        attrs = trial.user_attrs
        cabecalho = (
            f"Trial {trial.number + 1:02d}/{n_trials} | "
            f"{attrs.get('duracao_s', 0.0):.2f} s | {trial.state.name}"
        )

        if trial.value is None:
            print(cabecalho, flush=True)
            return

        por_horizonte = ", ".join(
            f"H{h}={mae:.6g}"
            for h, mae in zip(horizontes, attrs["mae_por_horizonte"], strict=True)
        )
        print(
            f"{cabecalho} | MAE validação={trial.value:.6g} | "
            f"{por_horizonte} | melhor trial={study.best_value:.6g}\n"
            f"  parâmetros={attrs['parametros_completos']}",
            flush=True,
        )

    study = optuna.create_study(
        direction="minimize",
        sampler=optuna.samplers.TPESampler(seed=seed),
    )
    study.optimize(
        objetivo,
        n_trials=n_trials,
        callbacks=[registrar_trial],
        show_progress_bar=True,
    )

    if study.best_value >= mae_medio_padrao:
        print(f"Selecionado LightGBM padrão | MAE={mae_medio_padrao:.6g}", flush=True)
        return tuning.DEFAULT_PARAMS, mae_padrao, mae_padrao, study

    best = study.best_trial
    print(
        f"Selecionado trial {best.number + 1} | MAE={study.best_value:.6g}",
        flush=True,
    )
    return (
        best.user_attrs["parametros_completos"],
        best.user_attrs["mae_por_horizonte"],
        mae_padrao,
        study,
    )


optuna.logging.set_verbosity(optuna.logging.WARNING)

partes_metricas = []
bundle = {"origem": None, "modelos": {}}
audit = None

with patch.object(tuning, "_optimize", otimizar_com_log):
    for variavel in TARGETS:
        print(f"\n=== {variavel} ===", flush=True)

        parte_bundle, parte_metricas, parte_audit = tuning.train_forecasts(
            history,
            [variavel],
            HORIZONTES,
            n_trials=TENTATIVAS_OPTUNA,
            seed=SEMENTE,
        )

        bundle["origem"] = parte_bundle["origem"]
        bundle["modelos"].update(parte_bundle["modelos"])
        partes_metricas.append(parte_metricas)

        if audit is None:
            audit = parte_audit
        else:
            audit["estudos"].update(parte_audit["estudos"])

if audit is None or not bundle["modelos"]:
    raise ValueError("Histórico insuficiente para gerar os modelos.")

metrics = pd.concat(partes_metricas, ignore_index=True)
metrics["locais_amostra"] = sample_size
metrics["semente_amostra"] = SEMENTE

audit["metrica_objetivo"] = "MAE médio de validação"
audit["espaco_busca"] = ESPACO_BUSCA

print(f"\nTreino: {audit['treino_inicio']} a {audit['treino_fim']}")
print(f"Validação: {audit['validacao']}")
print(f"Teste final: {audit['teste']}")

esperados = {(variavel, h) for variavel in TARGETS for h in HORIZONTES}
faltantes = sorted(esperados.difference(bundle["modelos"]))

if faltantes:
    print("\nPares sem modelo nesta rodada:")
    for variavel, horizonte in faltantes:
        print(f"  {variavel} | H{horizonte}")


=== temperatura__ar_2m_c ===
LightGBM padrão | MAE validação=2.0571


  0%|          | 0/24 [00:00<?, ?it/s]

Trial 01/24 | 1.05 s | COMPLETE | MAE validação=3.08563 | H1=1.29465, H2=2.58462, H3=5.37763 | melhor trial=3.08563
  parâmetros={'objective': 'regression_l1', 'boosting_type': 'gbdt', 'data_sample_strategy': 'goss', 'n_estimators': 220, 'learning_rate': 0.08006144509284034, 'max_depth': 3, 'num_leaves': 8, 'min_child_samples': 130, 'min_child_weight': 0.0070689749506246055, 'min_split_gain': 0.18182496720710062, 'reg_alpha': 0.9170225492671691, 'reg_lambda': 3.0424224295953772, 'max_bin': 63, 'min_data_in_bin': 20, 'colsample_bytree': 0.6988228512282146}


Best trial: 0. Best value: 3.08563:   4%|▍         | 1/24 [00:01<00:24,  1.05s/it]

Trial 02/24 | 3.75 s | COMPLETE | MAE validação=4.36063 | H1=2.5337, H2=3.64195, H3=6.90623 | melhor trial=3.08563
  parâmetros={'objective': 'fair', 'boosting_type': 'dart', 'data_sample_strategy': 'bagging', 'n_estimators': 220, 'learning_rate': 0.031995373905230294, 'max_depth': 5, 'num_leaves': 8, 'min_child_samples': 25, 'min_child_weight': 6.245139574743075, 'min_split_gain': 0.9656320330745594, 'reg_alpha': 4.041986740582305, 'reg_lambda': 3.0461376917337066, 'max_bin': 127, 'min_data_in_bin': 6, 'colsample_bytree': 0.8233119185389446, 'subsample_freq': 1, 'subsample': 0.7590988766312938}


Best trial: 0. Best value: 3.08563:   8%|▊         | 2/24 [00:04<00:58,  2.64s/it]

... (450 outputs left out)

Trial 24/24 | 0.90 s | COMPLETE | MAE validação=0.017996 | H1=0.012958, H2=0.0197509, H3=0.0212789 | melhor trial=0.017996
  parâmetros={'objective': 'regression_l1', 'boosting_type': 'gbdt', 'data_sample_strategy': 'goss', 'n_estimators': 280, 'learning_rate': 0.11819932232845255, 'max_depth': 3, 'num_leaves': 6, 'min_child_samples': 105, 'min_child_weight': 0.00987700268272427, 'min_split_gain': 0.48569469453213077, 'reg_alpha': 1.7814923879836555, 'reg_lambda': 0.38111861876715447, 'max_bin': 63, 'min_data_in_bin': 21, 'colsample_bytree': 0.7032313262050836}


Best trial: 23. Best value: 0.017996: 100%|██████████| 24/24 [00:24<00:00,  1.04s/it] 

Selecionado LightGBM padrão | MAE=0.0170249



Treino: 2019-01 a 2026-03
Validação: ['2026-04', '2026-05']
Teste final: ['2026-06', '2026-07', '2026-08']

Pares sem modelo nesta rodada:
  biologia__ndvi | H1
  biologia__ndvi | H2


## Salvar o modelo, as métricas e o histórico da busca

In [12]:
modelos = bundle["modelos"]
estudos = audit["estudos"]

if not modelos:
    raise ValueError("Execute a célula de otimização antes de salvar o pickle.")

for (variavel, horizonte), modelo in sorted(modelos.items()):
    if modelo is None:
        print(f"{variavel} | H{horizonte} | persistência")
        continue

    escolhidos = estudos[variavel]["parametros_selecionados"]
    ajustados = modelo.get_params()

    divergencias = {
        nome: {"selecionado": valor, "ajustado": ajustados.get(nome)}
        for nome, valor in escolhidos.items()
        if ajustados.get(nome) != valor
    }
    if divergencias:
        raise ValueError(
            f"Parâmetros divergentes para {variavel} em H{horizonte}: "
            f"{divergencias}. Execute novamente a célula de otimização."
        )

    print(f"{variavel} | H{horizonte} | LightGBM | {escolhidos}")

esperados = {(variavel, h) for variavel in TARGETS for h in HORIZONTES}
faltantes = sorted(esperados.difference(modelos))
if faltantes:
    print("\nPares ausentes no pickle. Confira a cobertura antes de publicar:")
    for variavel, horizonte in faltantes:
        print(f"  {variavel} | H{horizonte}")

model_dir = RUN / "modelos"
model_dir.mkdir(parents=True, exist_ok=True)
artifact = model_dir / "previsao_mensal.pkl"

with artifact.open("wb") as file:
    pickle.dump(bundle, file, protocol=pickle.HIGHEST_PROTOCOL)

metrics.to_parquet(RUN / "metricas_previsao.parquet", index=False)
(RUN / "otimizacao_lightgbm.json").write_text(
    json.dumps(audit, ensure_ascii=False, indent=2) + "\n",
    encoding="utf-8",
)

print(f"\nPickle salvo em {artifact}")

agua_gelo__precipitacao_mm_dia | H1 | LightGBM | {'objective': 'regression_l1', 'boosting_type': 'gbdt', 'data_sample_strategy': 'bagging', 'n_estimators': 280, 'learning_rate': 0.05583978798309175, 'max_depth': 6, 'num_leaves': 42, 'min_child_samples': 25, 'min_child_weight': 0.007655429821462699, 'min_split_gain': 0.8319696786044027, 'reg_alpha': 1.2526438271297853, 'reg_lambda': 2.8094316549528777, 'max_bin': 255, 'min_data_in_bin': 24, 'colsample_bytree': 0.7459961588018882, 'subsample_freq': 0}
agua_gelo__precipitacao_mm_dia | H2 | LightGBM | {'objective': 'regression_l1', 'boosting_type': 'gbdt', 'data_sample_strategy': 'bagging', 'n_estimators': 280, 'learning_rate': 0.05583978798309175, 'max_depth': 6, 'num_leaves': 42, 'min_child_samples': 25, 'min_child_weight': 0.007655429821462699, 'min_split_gain': 0.8319696786044027, 'reg_alpha': 1.2526438271297853, 'reg_lambda': 2.8094316549528777, 'max_bin': 255, 'min_data_in_bin': 24, 'colsample_bytree': 0.7459961588018882, 'subsample_

## Conferir o erro por horizonte

In [9]:
summary = [
    "variavel", "horizonte", "configuracao", "status", "n_teste",
    "mae_teste", "mae_teste_padrao", "mae_teste_persistencia", "f1_teste", "roc_auc_teste",
]
print(metrics.reindex(columns=summary).to_string(index=False))

                                          variavel  horizonte configuracao                 status  n_teste  mae_teste  mae_teste_padrao  mae_teste_persistencia  f1_teste  roc_auc_teste
                              temperatura__ar_2m_c          1       padrao               lightgbm   1200.0   1.174729          1.174729                2.489666  0.868895       0.995141
                              temperatura__ar_2m_c          2       padrao               lightgbm   1200.0   1.599544          1.599544                5.133583  0.823834       0.987802
                              temperatura__ar_2m_c          3       padrao               lightgbm   1200.0   1.968816          1.968816                9.076501  0.760638       0.980472
            temperatura__lst_amplitude_dia_noite_c          1       optuna               lightgbm   1182.0   1.134543          1.365403                1.445516  0.619048       0.992261
            temperatura__lst_amplitude_dia_noite_c          2       optuna 

In [13]:
import json

from IPython.display import display

nichos = json.loads((ROOT / "config" / "nichos.json").read_text(encoding="utf-8"))
selecao = json.loads((RUN / "selecao.json").read_text(encoding="utf-8"))
alvos_dinamicos = set(selecao["alvos_previsao"])

colunas_metricas = [
    "variavel", "horizonte", "status", "n_teste",
    "mae_teste", "mae_teste_padrao", "mae_teste_persistencia",
    "f1_teste", "recall_teste", "precisao_teste", "roc_auc_teste",
]
chaves = ["variavel", "horizonte"]

if "mae_teste" not in metrics:
    raise ValueError(
        "Execute a etapa de treino do Notebook 02 para gerar as métricas de teste."
    )

# Inclui Todos e conta cada variável uma vez dentro de cada nicho.
mapa = []
for corpo, grupos in nichos.items():
    variaveis_por_nicho = {
        nome: {item["terra"] for item in criterios}
        for nome, criterios in grupos.items()
    }
    variaveis_por_nicho["Todos"] = set().union(*variaveis_por_nicho.values())

    for nome, variaveis in variaveis_por_nicho.items():
        for variavel in sorted(variaveis):
            if variavel in alvos_dinamicos:
                for horizonte in HORIZONTES:
                    mapa.append({
                        "corpo": corpo,
                        "nicho": nome,
                        "variavel": variavel,
                        "horizonte": horizonte,
                    })

if not mapa:
    raise ValueError("Os nichos configurados estão sem variáveis dinâmicas avaliáveis.")

detalhe = pd.DataFrame(mapa).merge(
    metrics.reindex(columns=colunas_metricas),
    on=chaves,
    how="left",
    validate="many_to_one",
)

detalhe["avaliada"] = detalhe["mae_teste"].notna()
detalhe["status"] = detalhe["status"].fillna("sem resultado")

# Primeiro normaliza o erro por variável. Só depois calcula a média do nicho.
mae_ref = detalhe["mae_teste_persistencia"].where(
    detalhe["mae_teste_persistencia"] > 0
)
detalhe["ganho_mae_pct"] = 100 * (1 - detalhe["mae_teste"] / mae_ref)

resumo = (
    detalhe.groupby(["corpo", "nicho", "horizonte"], sort=True)
    .agg(
        variaveis_esperadas=("variavel", "nunique"),
        variaveis_avaliadas=("avaliada", "sum"),
        ganho_mae_pct=("ganho_mae_pct", "mean"),
        f1=("f1_teste", "mean"),
        recall=("recall_teste", "mean"),
        precisao=("precisao_teste", "mean"),
        roc_auc=("roc_auc_teste", "mean"),
    )
    .reset_index()
)

resumo["cobertura"] = (
    resumo["variaveis_avaliadas"].astype(str)
    + "/"
    + resumo["variaveis_esperadas"].astype(str)
)

print("Resumo do teste por corpo, nicho e horizonte")
display(
    resumo[
        ["corpo", "nicho", "horizonte", "cobertura",
         "ganho_mae_pct", "f1", "recall", "precisao", "roc_auc"]
    ].round(3)
)

# Para conferir um grupo, ajuste estes dois filtros.
CORPO_DETALHE = "lua"
NICHO_DETALHE = "Todos"

dicionario = pd.read_csv(
    TERRA / "spec_analogos" / "dicionario_spec.csv",
    usecols=["coluna", "unidade"],
)
unidades = dicionario.set_index("coluna")["unidade"]
detalhe["unidade"] = detalhe["variavel"].map(unidades)

print(f"\nDetalhe — {CORPO_DETALHE} / {NICHO_DETALHE}")
display(
    detalhe.loc[
        detalhe["corpo"].eq(CORPO_DETALHE)
        & detalhe["nicho"].eq(NICHO_DETALHE),
        [
            "variavel", "horizonte", "unidade", "status", "n_teste",
            "mae_teste", "mae_teste_padrao", "mae_teste_persistencia",
            "ganho_mae_pct", "f1_teste", "recall_teste",
            "precisao_teste", "roc_auc_teste",
        ],
    ].sort_values(chaves).round(4)
)

Resumo do teste por corpo, nicho e horizonte


,corpo,nicho,horizonte,cobertura,ganho_mae_pct,f1,recall,precisao,roc_auc
0,lua,Todos,1,5/6,42.583,0.771,0.747,0.805,0.982
1,lua,Todos,2,5/6,50.948,0.717,0.703,0.751,0.970
2,lua,Todos,3,6/6,53.987,0.639,0.595,0.770,0.950
3,lua,agua_gelo,1,1/1,17.458,0.580,0.516,0.663,0.935
4,lua,agua_gelo,2,1/1,24.162,0.538,0.452,0.663,0.897
5,lua,agua_gelo,3,1/1,25.980,0.406,0.317,0.563,0.854
6,lua,iluminacao_radiacao,1,1/1,47.324,0.882,0.882,0.882,0.990
7,lua,iluminacao_radiacao,2,1/1,58.756,0.838,0.826,0.850,0.979
8,lua,iluminacao_radiacao,3,1/1,68.497,0.784,0.771,0.799,0.972
9,lua,superficie_poeira,1,1/2,73.803,0.903,0.875,0.933,0.999



Detalhe — lua / Todos


,variavel,horizonte,unidade,status,n_teste,mae_teste,mae_teste_padrao,mae_teste_persistencia,ganho_mae_pct,f1_teste,recall_teste,precisao_teste,roc_auc_teste
18,agua_gelo__precipitacao_mm_dia,1,mm/dia,lightgbm,1200.0,0.7602,0.7945,0.9210,17.4583,0.5804,0.5159,0.6633,0.9347
19,agua_gelo__precipitacao_mm_dia,2,mm/dia,lightgbm,1200.0,0.8646,0.9364,1.1401,24.1621,0.5377,0.4524,0.6628,0.8970
20,agua_gelo__precipitacao_mm_dia,3,mm/dia,lightgbm,1200.0,1.0199,1.1642,1.3779,25.9797,0.4061,0.3175,0.5634,0.8538
21,biologia__ndvi,1,−1–1,historico insuficiente,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
22,biologia__ndvi,2,−1–1,historico insuficiente,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
23,biologia__ndvi,3,−1–1,lightgbm,479.0,0.0774,0.0774,0.1616,52.0973,0.3910,0.2500,0.8966,0.9083
24,geologia_superficie__albedo,1,0–1,lightgbm,852.0,0.0057,0.0057,0.0217,73.8033,0.9032,0.8750,0.9333,0.9988
25,geologia_superficie__albedo,2,0–1,lightgbm,822.0,0.0086,0.0086,0.0505,82.9952,0.8182,0.8182,0.8182,0.9993
26,geologia_superficie__albedo,3,0–1,lightgbm,761.0,0.0119,0.0119,0.0717,83.3769,0.9412,0.8889,1.0000,0.9999
27,iluminacao_radiacao__radiacao_solar_superficie...,1,W/m²,lightgbm,831.0,14.3632,14.3632,27.2671,47.3241,0.8819,0.8819,0.8819,0.9903


A inferência em lote lê apenas os meses necessários no Parquet observado.
Ao atualizar o mês de corte, regenere o artefato e revise as métricas.
O mapa lê o Parquet publicado.

## Recarregar o modelo e publicar as previsões

In [14]:
cutoff = pd.Period(bundle["origem"], freq="M")
lag_months = [int((cutoff - lag).strftime("%Y%m")) for lag in LAGS]
recent = pd.read_parquet(
    terra_path, columns=["location_id", "anomes", *TARGETS],
    filters=[("anomes", "in", lag_months)],
)
forecasts = publish_forecasts(recent, artifact)
if forecasts.empty:
    raise ValueError("Nenhum local possui as entradas necessárias no mês de corte.")
forecasts.to_parquet(RUN / "previsoes.parquet", index=False)
print(f"{len(forecasts):,} previsões para {forecasts['location_id'].nunique():,} locais")

1,885,320 previsões para 85,726 locais


## Conferir os valores previstos

In [15]:
print(forecasts.head().to_string(index=False))

location_id  anomes             variavel  valor_previsto
    T000001  202609 temperatura__ar_2m_c       -8.793247
    T000002  202609 temperatura__ar_2m_c      -10.123420
    T000003  202609 temperatura__ar_2m_c      -10.123420
    T000004  202609 temperatura__ar_2m_c      -10.123420
    T000005  202609 temperatura__ar_2m_c      -10.123420
